# Atividade Prática — Regressão Linear Simples, Múltipla e Polinomial
## Preço médio de imóveis nos distritos de Londres

**Nome:** Felipe Estevo Freitas • **RA:** 1990153 • **Turma:** 4Termo IA • **Disciplina:** Aprendizado de Máquina
**Ambiente:** Jupyter / Google Colab • **Bibliotecas:** NumPy, pandas, Matplotlib e scikit-learn

### Problema
Prever o **preço médio de venda de imóveis (£) de um distrito de Londres em dezembro**, usando indicadores socioeconômicos do distrito. É regressão porque o alvo é uma quantidade contínua.

### Dataset
- **Título:** Housing in London • **Publicador:** Justinas Cirtautas (Kaggle) • **Dados reais** (origem: London Datastore / GLA)
- **URL:** https://www.kaggle.com/datasets/justinas/housing-in-london •
- **Arquivos:** `housing_in_london_monthly_variables.csv` e `housing_in_london_yearly_variables.csv`
- **Uma linha** = um distrito em um ano (dez/1999 a dez/2019).
**Reprodução:** execute a partir da raiz do repositório. Dados locais em `data/` ou, com autorização de acesso à fonte, obtenha-os usando `python scripts/prepare_data.py --download`. Em CI, o notebook é executado do zero; as métricas são exportadas para `artifacts/`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Versão do scikit-learn:", sklearn.__version__)
print("Versão do pandas:", pd.__version__, "| numpy:", np.__version__)

# Dados locais são usados primeiro; download exige autorização explícita.
from pathlib import Path
from scripts.prepare_data import ensure_data

DATA_DIR = ensure_data(allow_download=os.environ.get("LONDON_DATA_AUTO_DOWNLOAD", "0") == "1")
print("Diretório dos CSVs:", DATA_DIR)

## 1. Carregamento e limpeza
Unimos o preço de dezembro (arquivo mensal) aos indicadores anuais e mantemos apenas distritos reais (`borough_flag == 1`; os demais são regiões como "england").

- **Alvo:** `average_price` (£) • **Preditor principal:** `median_salary` (£/ano)
- **Outros preditores:** `number_of_jobs` (empregos) e `recycling_pct` (% reciclado)
- Valores ausentes nos preditores serão imputados **dentro do Pipeline**; não há ausentes no alvo.

In [ ]:
mensal = pd.read_csv(DATA_DIR / "housing_in_london_monthly_variables.csv", parse_dates=["date"])
anual = pd.read_csv(DATA_DIR / "housing_in_london_yearly_variables.csv", parse_dates=["date"])

# Preço de dezembro de cada distrito (os indicadores anuais são de dezembro).
mensal = mensal[(mensal.borough_flag == 1) & (mensal.date.dt.month == 12)]
mensal = mensal[["area", "date", "average_price"]]
df = anual[anual.borough_flag == 1].merge(mensal, on=["area", "date"])

ALVO = "average_price"
X1 = ["median_salary"]                                        # preditor da simples e da polinomial
PREDITORES = ["median_salary", "number_of_jobs", "recycling_pct"]  # preditores da múltipla

# Algumas colunas numéricas vieram como texto ('#', 'na'): viram NaN.
for c in PREDITORES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[ALVO]).drop_duplicates()
df = df[["area", "date", ALVO] + PREDITORES].reset_index(drop=True)

print("Dimensões:", df.shape, "| distritos:", df.area.nunique())
print("\nValores ausentes:")
print(df.isna().sum())

## 2. Separação treino / validação / teste
Os dados são repetidos por distrito ao longo dos anos, então separamos **por distrito**: um mesmo distrito nunca aparece em dois conjuntos. Primeiro reservamos 20% para teste; depois 25% do restante para validação (≈ 60/20/20), sempre com `random_state=42`.

> **Pergunta:** por que não sortear linhas aleatoriamente?
> Porque linhas do mesmo distrito são parecidas entre si; o teste ficaria "fácil demais".

In [ ]:
distritos = df.area.unique()

resto, d_teste = train_test_split(distritos, test_size=0.20, random_state=42)
d_treino, d_val = train_test_split(resto, test_size=0.25, random_state=42)

treino = df[df.area.isin(d_treino)].copy()
val = df[df.area.isin(d_val)].copy()
teste = df[df.area.isin(d_teste)].copy()
treino_val = pd.concat([treino, val])   # usado só no reajuste final

# Integridade: nenhum distrito pode atravessar as partições.
assert not (set(d_treino) & set(d_val) or set(d_treino) & set(d_teste) or set(d_val) & set(d_teste))
assert len(treino) + len(val) + len(teste) == len(df)

print("Registros -> treino:", len(treino), "| validação:", len(val), "| teste:", len(teste))

## 3. Exploração dos dados (somente treino)

In [ ]:
print(treino[[ALVO] + PREDITORES].describe().round(1))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].hist(treino[ALVO], bins=30)
ax[0].set(title="Distribuição do alvo (treino)", xlabel="Preço médio (£)", ylabel="Frequência")

ax[1].scatter(treino.median_salary, treino[ALVO], s=10)
ax[1].set(title="Salário mediano × preço (treino)", xlabel="Salário mediano (£/ano)", ylabel="Preço médio (£)")

corr = treino[[ALVO] + PREDITORES].corr()
ax[2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax[2].set_xticks(range(len(corr))); ax[2].set_xticklabels(corr.columns, rotation=45, ha="right")
ax[2].set_yticks(range(len(corr))); ax[2].set_yticklabels(corr.columns)
ax[2].set_title("Correlação (treino)")
for i in range(len(corr)):
    for j in range(len(corr)):
        ax[2].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()

**Hipóteses:** o preço sobe com o salário (relação positiva); a distribuição do preço é assimétrica à direita, com alguns distritos muito caros (valores extremos); salário e empregos são preditores parcialmente redundantes.

## 4. Funções auxiliares
- `Pipeline` encadeia as etapas; cada uma aprende **somente no treino** (evita vazamento).
- `SimpleImputer` preenche ausentes com a mediana; `StandardScaler` padroniza.
- `PolynomialFeatures(include_bias=False)` cria x, x², x³… (só **transforma** a entrada). Com `grau=1` ele não altera nada: é a regressão linear comum.
- `LinearRegression.fit` aprende os coeficientes que minimizam a soma dos erros ao quadrado; `predict` aplica a equação aprendida.

In [ ]:
def criar_modelo(grau=1):
    """
    Regressão linear com atributos polinomiais.
    grau=1 -> regressão linear (simples ou múltipla, conforme as colunas usadas).
    """
    return Pipeline([
        ("imputador", SimpleImputer(strategy="median")),
        ("escala", StandardScaler()),
        ("polinomio", PolynomialFeatures(degree=grau, include_bias=False)),
        ("modelo", LinearRegression())
    ])


def calcular_metricas(modelo, X_treino, y_treino, X_aval, y_aval):
    """Treina o modelo e calcula MAE, RMSE e R² em treino e validação."""
    modelo.fit(X_treino, y_treino)
    p_treino = modelo.predict(X_treino)
    p_aval = modelo.predict(X_aval)

    return {
        "MAE treino": mean_absolute_error(y_treino, p_treino),
        "RMSE treino": np.sqrt(mean_squared_error(y_treino, p_treino)),
        "R² treino": r2_score(y_treino, p_treino),
        "MAE val": mean_absolute_error(y_aval, p_aval),
        "RMSE val": np.sqrt(mean_squared_error(y_aval, p_aval)),
        "R² val": r2_score(y_aval, p_aval),
    }


def mostrar_modelo(modelos, titulo):
    """Dispersão (treino e validação) com a curva de cada modelo sobre uma grade ordenada de X."""
    grade = pd.DataFrame({"median_salary": np.linspace(treino.median_salary.min(),
                                                       treino.median_salary.max(), 300)})
    plt.figure(figsize=(10, 5))
    plt.scatter(treino.median_salary, treino[ALVO], alpha=0.4, s=12, label="Treino")
    plt.scatter(val.median_salary, val[ALVO], marker="x", alpha=0.8, label="Validação")
    for nome, modelo in modelos.items():
        plt.plot(grade, modelo.predict(grade), linewidth=2, label=nome)
    plt.xlabel("Salário mediano (£/ano)")
    plt.ylabel("Preço médio (£)")
    plt.title(titulo)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

## 5. Baseline e os três cenários
O baseline (`DummyRegressor`) sempre prevê a **média do treino**: é a referência mínima que qualquer modelo precisa superar.

> **Antes de executar, pense:** a regressão simples vai superar o baseline? Adicionar variáveis ajuda? E aumentar o grau?

In [ ]:
modelos = {
    "Baseline":    (X1, DummyRegressor(strategy="mean")),
    "Simples":     (X1, criar_modelo(grau=1)),
    "Múltipla":    (PREDITORES, criar_modelo(grau=1)),
    "Poli grau 2": (X1, criar_modelo(grau=2)),
    "Poli grau 3": (X1, criar_modelo(grau=3)),
    "Poli grau 5": (X1, criar_modelo(grau=5)),
}

linhas = {}
for nome, (cols, modelo) in modelos.items():
    linhas[nome] = calcular_metricas(modelo, treino[cols], treino[ALVO], val[cols], val[ALVO])

tabela = pd.DataFrame(linhas).T
print("Métricas em £ (R² sem unidade) — modelos ajustados só no treino")
Path("artifacts").mkdir(exist_ok=True)
tabela.to_csv("artifacts/metricas_validacao.csv", index_label="modelo", float_format="%.6f")
tabela.round(3)

In [ ]:
# Como padronizamos, convertemos: coef_original = coef_padronizado / desvio_padrão
for nome, cols in [("Simples", X1), ("Múltipla", PREDITORES)]:
    m = modelos[nome][1]
    coef = m["modelo"].coef_ / m["escala"].scale_
    intercepto = m["modelo"].intercept_ - np.sum(coef * m["escala"].mean_)
    print(f"{nome}: intercepto = {intercepto:,.0f} £")
    for c, k in zip(cols, coef):
        print(f"    {c}: {k:,.2f}")

In [ ]:
# Reta (grau 1) e curvas dos graus 2, 3 e 5.
mostrar_modelo(
    {n: modelos[n][1] for n in ["Simples", "Poli grau 2", "Poli grau 3", "Poli grau 5"]},
    "Ajustes sobre o salário mediano"
)

# RMSE em função do grau (incluindo o grau 1), antes do reajuste final.
graus = {1: "Simples", 2: "Poli grau 2", 3: "Poli grau 3", 5: "Poli grau 5"}

plt.figure(figsize=(7, 4))
plt.plot(list(graus), [tabela.loc[n, "RMSE treino"] for n in graus.values()], marker="o", label="Treino")
plt.plot(list(graus), [tabela.loc[n, "RMSE val"] for n in graus.values()], marker="s", label="Validação")
plt.xlabel("Grau do polinômio")
plt.ylabel("RMSE (£)")
plt.title("RMSE por grau")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### O que observar
- O erro de **treino** cai conforme o grau aumenta, mas o de **validação** sobe: o modelo mais flexível ajusta melhor o que já viu e generaliza pior.
- Só a regressão **múltipla** supera o baseline na validação.

## 6. Escolha do grau pela validação (o teste ainda não foi usado)

In [ ]:
grau_escolhido = int(min([2, 3, 5], key=lambda g: tabela.loc[f"Poli grau {g}", "RMSE val"]))
print("Grau escolhido (menor RMSE de validação):", grau_escolhido)

## 7. Reajuste final e avaliação no teste
Com as escolhas fechadas, reajustamos baseline, simples, múltipla e o polinômio escolhido em **treino + validação** e avaliamos no teste **uma única vez**. O baseline passa a usar a média de treino + validação (nunca a do teste).

In [ ]:
finais = {
    "Baseline": (X1, DummyRegressor(strategy="mean")),
    "Linear simples": (X1, criar_modelo(grau=1)),
    "Linear múltipla": (PREDITORES, criar_modelo(grau=1)),
    f"Polinomial grau {grau_escolhido}": (X1, criar_modelo(grau=grau_escolhido)),
}

previsoes, resultados = {}, {}
for nome, (cols, modelo) in finais.items():
    modelo.fit(treino_val[cols], treino_val[ALVO])
    previsoes[nome] = modelo.predict(teste[cols])
    resultados[nome] = {
        "MAE teste": mean_absolute_error(teste[ALVO], previsoes[nome]),
        "RMSE teste": np.sqrt(mean_squared_error(teste[ALVO], previsoes[nome])),
        "R² teste": r2_score(teste[ALVO], previsoes[nome]),
    }

final = pd.DataFrame(resultados).T
print("Métricas finais no teste (£)")
Path("artifacts").mkdir(exist_ok=True)
final.to_csv("artifacts/metricas_teste.csv", index_label="modelo", float_format="%.6f")
final.round(3)

In [ ]:
# Valores reais × previstos (linha vermelha = previsão perfeita)
fig, ax = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
lim = [teste[ALVO].min(), teste[ALVO].max()]
for a, (nome, p) in zip(ax, previsoes.items()):
    a.scatter(teste[ALVO], p, s=10)
    a.plot(lim, lim, "r--", label="Previsão perfeita")
    a.set(title=nome, xlabel="Real (£)")
ax[0].set_ylabel("Previsto (£)")
ax[0].legend()
plt.tight_layout()
plt.show()

# Resíduos (real − previsto) × previsto
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, nome in zip(ax, list(previsoes)[1:]):
    a.scatter(previsoes[nome], teste[ALVO] - previsoes[nome], s=10)
    a.axhline(0, color="r", linestyle="--")
    a.set(title=nome, xlabel="Previsto (£)")
ax[0].set_ylabel("Resíduo (£)")
plt.tight_layout()
plt.show()

In [ ]:
nome = "Linear múltipla"
exemplos = teste[["area", "date", ALVO]].copy()
exemplos["previsto"] = previsoes[nome]
exemplos["residuo"] = exemplos[ALVO] - exemplos["previsto"]

# Menor e maior erro absoluto
exemplos = exemplos.reindex(exemplos["residuo"].abs().sort_values().index)
pd.concat([exemplos.head(1), exemplos.tail(1)]).round({"previsto": 0, "residuo": 0})

## 8. Diagnóstico e conclusão
- **Validação:** só a regressão **múltipla** superou o baseline (RMSE £107,9 mil vs £136,1 mil; R² = 0,10). Simples e polinomiais ficaram **piores** que o baseline, e o erro de validação **sobe com o grau** enquanto o de treino cai — sinal de sobreajuste. O grau 2 teve o menor RMSE de validação entre 2, 3 e 5.
- **Teste:** todos os modelos ficaram perto do baseline (R² entre 0,04 e 0,10; RMSE £249–257 mil vs £271 mil). A múltipla, melhor na validação, **não** se sustentou no teste — resultado inesperado, relatado sem alterar as escolhas.
- **Por quê:** o teste contém distritos muito caros (ex.: Kensington and Chelsea, resíduo de mais de £1 milhão no exemplo da célula 12). O RMSE, muito maior que o MAE, é dominado por esses erros, e o gráfico de resíduos mostra formato de **funil** (erro cresce com o preço previsto). Com só 33 distritos, as métricas são instáveis.
- **Limitações:** salário não explica sozinho o preço (localização, tipo de imóvel e inflação ficam de fora); preços e salários são nominais e crescem no tempo; extrapolar para distritos de preço extremo é arriscado.



> **Uso restrito a fins didáticos:** dados de preços médios agregados por distrito não fornecem estimativas confiáveis para precificação de imóveis individuais. Os resultados não representam validação prospectiva no tempo nem desempenho comprovado para uso comercial.
